# NIDS Full Experiment Suite — CIC-IDS Collection
## Self-Running | Zero Intervention Required

**Experiments covered:**
1. Data loading, cleaning, EDA
2. Baseline models (LightGBM, XGBoost, RandomForest, ExtraTrees)
3. CatBoost (if installed)
4. Flat vs Hierarchical pipeline
5. Class imbalance strategies (scale_pos_weight, balanced, SMOTE)
6. Threshold tuning (0.5 vs PR-curve optimized)
7. Optuna hyperparameter tuning (LightGBM)
8. Feature ablation (causal-only features)
9. Probability calibration for rare classes
10. Inference speed benchmark
11. **Master results table + conclusion**

> **How to run:** Kernel → Restart & Run All. Come back when done.

In [ ]:
# ── CELL 1: Install / verify dependencies ──────────────────────────────────
import subprocess, sys

PKGS = [
    'lightgbm', 'xgboost', 'optuna', 'imbalanced-learn',
    'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'pyarrow',
    'kagglehub'
]
for pkg in PKGS:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Ensure KaggleHub pandas adapter extras are available
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub[pandas-datasets]', '-q'])

try:
    import catboost
    HAS_CATBOOST = True
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost', '-q'])
    try:
        import catboost
        HAS_CATBOOST = True
    except Exception:
        HAS_CATBOOST = False
        print('CatBoost unavailable — will skip those experiments')

print('All dependencies ready.')

In [ ]:
# ── CELL 2: Imports & global config ────────────────────────────────────────
import warnings, time, os, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, classification_report, precision_recall_curve
)
from imblearn.over_sampling import SMOTE

# ── Config ──────────────────────────────────────────────────────────────────
KAGGLE_DATASET = "dhoogla/cicidscollection"
KAGGLE_FILE_PATH = "cic-collection.parquet"

RANDOM_STATE  = 42
TARGET_RECALL = 0.95        # minimum recall we want to guarantee
OPTUNA_TRIALS = 40          # increase for better tuning (40 is fast enough)
SMOTE_SUBSAMPLE = 300_000   # rows to subsample before SMOTE (memory guard)
RESULTS = {}                # master results dict — filled throughout

np.random.seed(RANDOM_STATE)
print('Config ready.')

---
## Section 1 — Data Loading & Preprocessing

In [ ]:
# ── CELL 3: Load data from KaggleHub ───────────────────────────────────────
import kagglehub
from kagglehub import KaggleDatasetAdapter

print('Loading dataset from KaggleHub...')
t0 = time.time()

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    KAGGLE_DATASET,
    KAGGLE_FILE_PATH
)

print(f'Loaded {len(df):,} rows × {len(df.columns)} cols in {time.time()-t0:.1f}s')
df.head(3)

In [ ]:
# ── CELL 4: Normalize column names ─────────────────────────────────────────
df.columns = (
    df.columns.str.strip().str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)
print('Columns normalized. Shape:', df.shape)

In [ ]:
# ── CELL 5: Clean infinities / NaNs / constants ─────────────────────────────
rows_before = len(df)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f'Dropped {rows_before - len(df):,} rows with inf/NaN')

LABEL_COLS = ['label', 'classlabel']
constant_cols = [
    c for c in df.columns
    if df[c].nunique() <= 1 and c not in LABEL_COLS
]
df.drop(columns=constant_cols, inplace=True)
print(f'Dropped {len(constant_cols)} constant columns. Remaining: {len(df.columns)}')

In [ ]:
# ── CELL 6: Memory optimization ─────────────────────────────────────────────
int_cols   = df.select_dtypes(['int64','int32']).columns.drop(LABEL_COLS, errors='ignore')
float_cols = df.select_dtypes(['float64','float32']).columns

for c in int_cols:
    mx, mn = df[c].max(), df[c].min()
    df[c] = df[c].astype('int8') if mx <= 127 and mn >= -128 else df[c].astype('int32')

df[float_cols] = df[float_cols].astype('float32')
print('Memory optimized.')
df.info(memory_usage='deep', verbose=False)

In [ ]:
# ── CELL 7: EDA — class distributions ──────────────────────────────────────
print('=== Multi-Class Distribution (classlabel) ===')
class_dist = df['classlabel'].value_counts()
print(class_dist.to_string())
print(f'\nClass balance (%):\n{(class_dist / len(df) * 100).round(2).to_string()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_dist.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class Counts (linear)')
axes[0].tick_params(axis='x', rotation=45)

class_dist.plot(kind='bar', ax=axes[1], color='steelblue', logy=True)
axes[1].set_title('Class Counts (log scale)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=120)
plt.show()
print('EDA plot saved.')

In [ ]:
# ── CELL 8: Label encoding ──────────────────────────────────────────────────
le = LabelEncoder()
df['multi_label']  = le.fit_transform(df['classlabel'])
df['binary_label'] = df['classlabel'].apply(lambda x: 0 if x == 'Benign' else 1)

CLASS_NAMES = le.classes_.tolist()
N_CLASSES   = len(CLASS_NAMES)
print(f'{N_CLASSES} classes:', CLASS_NAMES)
print(df[['classlabel','multi_label','binary_label']].head(5))

In [ ]:
# ── CELL 9: Train / Val / Test split (70/15/15 stratified) ─────────────────
X = df.drop(['label','classlabel','binary_label','multi_label'], axis=1, errors='ignore')
y_bin   = df['binary_label']
y_multi = df['multi_label']

X_train, X_temp, y_train_bin, y_temp_bin = train_test_split(
    X, y_bin, test_size=0.30, stratify=y_bin, random_state=RANDOM_STATE
)
X_val, X_test, y_val_bin, y_test_bin = train_test_split(
    X_temp, y_temp_bin, test_size=0.50, stratify=y_temp_bin, random_state=RANDOM_STATE
)

y_train_multi = y_multi.loc[X_train.index]
y_val_multi   = y_multi.loc[X_val.index]
y_test_multi  = y_multi.loc[X_test.index]

FEATURE_NAMES = X_train.columns.tolist()
X_train_np = X_train.values.astype(np.float32)
X_val_np   = X_val.values.astype(np.float32)
X_test_np  = X_test.values.astype(np.float32)

pos_weight = (y_train_bin == 0).sum() / (y_train_bin == 1).sum()

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Binary pos_weight (class imbalance): {pos_weight:.3f}')
print(f'Binary train distribution:\n{y_train_bin.value_counts(normalize=True).round(4) * 100}')

---
## Section 2 — Helper Functions

In [ ]:
# ── CELL 10: Shared helpers ─────────────────────────────────────────────────

def tune_threshold(y_true, y_prob, target_recall=TARGET_RECALL):
    """Pick threshold maximizing precision while satisfying recall >= target."""
    p, r, t = precision_recall_curve(y_true, y_prob)
    df_pr = pd.DataFrame({'threshold': t, 'precision': p[:-1], 'recall': r[:-1]})
    feasible = df_pr[df_pr['recall'] >= target_recall]
    if not feasible.empty:
        row = feasible.sort_values(['precision','threshold'], ascending=[False,False]).iloc[0]
    else:
        df_pr['f1'] = 2*df_pr['precision']*df_pr['recall'] / (df_pr['precision']+df_pr['recall']+1e-12)
        row = df_pr.sort_values('f1', ascending=False).iloc[0]
    return float(row['threshold'])


def eval_binary(y_true, y_prob, threshold=None, label=''):
    """Evaluate binary classifier. Returns metrics dict."""
    if threshold is None:
        threshold = tune_threshold(y_true, y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'AUC':       round(roc_auc_score(y_true, y_prob), 5),
        'F1':        round(f1_score(y_true, y_pred), 5),
        'Recall':    round(recall_score(y_true, y_pred), 5),
        'Precision': round(precision_score(y_true, y_pred), 5),
        'Threshold': round(threshold, 4),
    }
    if label:
        print(f'\n[{label}] Binary Test Metrics')
        for k, v in metrics.items(): print(f'  {k}: {v}')
    return metrics


def eval_multi(y_true, y_pred, label=''):
    """Evaluate multi-class classifier. Returns metrics dict."""
    metrics = {
        'Macro_F1':    round(f1_score(y_true, y_pred, average='macro'), 5),
        'Weighted_F1': round(f1_score(y_true, y_pred, average='weighted'), 5),
    }
    if label:
        print(f'\n[{label}] Multi-Class Test Metrics')
        for k, v in metrics.items(): print(f'  {k}: {v}')
        print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
    return metrics


def benchmark_speed(predict_fn, X_np, n_iter=5, label=''):
    """Measure inference throughput (predictions/sec)."""
    times = []
    for _ in range(n_iter):
        t0 = time.perf_counter()
        predict_fn(X_np)
        times.append(time.perf_counter() - t0)
    avg = np.mean(times)
    pps = len(X_np) / avg
    if label: print(f'  [{label}] Throughput: {pps:,.0f} pred/sec (avg over {n_iter} runs)')
    return round(pps, 0)


def compute_class_weights_dict(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes.tolist(), weights.tolist()))


print('Helpers defined.')

---
## Section 3 — Experiment 1: LightGBM Defaults (Binary + Multi)

In [ ]:
# ── CELL 11: LightGBM Default — Binary ─────────────────────────────────────
print('=== EXP 1a: LightGBM Default — Binary ===')

lgb_bin_params = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 64,
    'min_child_samples': 20, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'scale_pos_weight': pos_weight,
    'device': 'cpu', 'verbose': -1, 'seed': RANDOM_STATE
}

lgb_bin_train = lgb.Dataset(X_train_np, label=y_train_bin, feature_name=FEATURE_NAMES)
lgb_bin_val   = lgb.Dataset(X_val_np, label=y_val_bin, reference=lgb_bin_train)

t0 = time.time()
lgb_bin_model = lgb.train(
    lgb_bin_params, lgb_bin_train, num_boost_round=2000,
    valid_sets=[lgb_bin_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0
print(f'Training time: {train_time:.1f}s')

lgb_bin_val_prob  = lgb_bin_model.predict(X_val_np)
lgb_bin_threshold = tune_threshold(y_val_bin, lgb_bin_val_prob)
lgb_bin_test_prob = lgb_bin_model.predict(X_test_np)
metrics = eval_binary(y_test_bin, lgb_bin_test_prob, lgb_bin_threshold, 'LightGBM-Default-Binary')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(lgb_bin_model.predict, X_test_np, label='LightGBM-Binary')
RESULTS['LGB_Default_Binary'] = metrics

In [ ]:
# ── CELL 12: LightGBM Default — Multi-Class ────────────────────────────────
print('=== EXP 1b: LightGBM Default — Multi-Class ===')

cw = compute_class_weights_dict(y_train_multi)
sw_train = y_train_multi.map(cw).to_numpy(dtype=np.float32)
sw_val   = y_val_multi.map(cw).to_numpy(dtype=np.float32)

lgb_multi_params = {
    'objective': 'multiclass', 'num_class': N_CLASSES,
    'metric': 'multi_logloss', 'learning_rate': 0.05,
    'num_leaves': 128, 'min_child_samples': 20,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'device': 'cpu', 'verbose': -1, 'seed': RANDOM_STATE
}

lgb_multi_train = lgb.Dataset(X_train_np, label=y_train_multi, weight=sw_train, feature_name=FEATURE_NAMES)
lgb_multi_val   = lgb.Dataset(X_val_np, label=y_val_multi, weight=sw_val, reference=lgb_multi_train)

t0 = time.time()
lgb_multi_model = lgb.train(
    lgb_multi_params, lgb_multi_train, num_boost_round=2000,
    valid_sets=[lgb_multi_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0
print(f'Training time: {train_time:.1f}s')

lgb_multi_prob = lgb_multi_model.predict(X_test_np)
lgb_multi_pred = np.argmax(lgb_multi_prob, axis=1)
metrics = eval_multi(y_test_multi, lgb_multi_pred, 'LightGBM-Default-Multi')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(
    lambda x: np.argmax(lgb_multi_model.predict(x), axis=1), X_test_np, label='LightGBM-Multi'
)
RESULTS['LGB_Default_Multi'] = metrics

---
## Section 4 — Experiment 2: XGBoost Defaults

In [ ]:
# ── CELL 13: XGBoost Default — Binary ──────────────────────────────────────
print('=== EXP 2a: XGBoost Default — Binary ===')

xgb_bin_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'learning_rate': 0.05, 'max_depth': 8,
    'min_child_weight': 20, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'scale_pos_weight': pos_weight,
    'tree_method': 'hist', 'device': 'cpu',
    'seed': RANDOM_STATE, 'verbosity': 0
}

dtrain = xgb.DMatrix(X_train_np, label=y_train_bin, feature_names=FEATURE_NAMES)
dval   = xgb.DMatrix(X_val_np, label=y_val_bin, feature_names=FEATURE_NAMES)
dtest  = xgb.DMatrix(X_test_np, label=y_test_bin, feature_names=FEATURE_NAMES)

t0 = time.time()
xgb_bin_model = xgb.train(
    xgb_bin_params, dtrain, num_boost_round=2000,
    evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=200
)
train_time = time.time() - t0
print(f'Training time: {train_time:.1f}s')

xgb_bin_val_prob  = xgb_bin_model.predict(dval)
xgb_bin_threshold = tune_threshold(y_val_bin, xgb_bin_val_prob)
xgb_bin_test_prob = xgb_bin_model.predict(dtest)
metrics = eval_binary(y_test_bin, xgb_bin_test_prob, xgb_bin_threshold, 'XGBoost-Default-Binary')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(xgb_bin_model.predict, dtest, label='XGBoost-Binary')
RESULTS['XGB_Default_Binary'] = metrics

In [ ]:
# ── CELL 14: XGBoost Default — Multi-Class ─────────────────────────────────
print('=== EXP 2b: XGBoost Default — Multi-Class ===')

xgb_multi_params = {
    'objective': 'multi:softprob', 'num_class': N_CLASSES,
    'eval_metric': 'mlogloss', 'learning_rate': 0.05,
    'max_depth': 8, 'min_child_weight': 20,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'tree_method': 'hist', 'device': 'cpu',
    'seed': RANDOM_STATE, 'verbosity': 0
}

dtrain_m = xgb.DMatrix(X_train_np, label=y_train_multi, feature_names=FEATURE_NAMES, weight=sw_train)
dval_m   = xgb.DMatrix(X_val_np, label=y_val_multi, feature_names=FEATURE_NAMES, weight=sw_val)
dtest_m  = xgb.DMatrix(X_test_np, label=y_test_multi, feature_names=FEATURE_NAMES)

t0 = time.time()
xgb_multi_model = xgb.train(
    xgb_multi_params, dtrain_m, num_boost_round=2000,
    evals=[(dval_m, 'val')], early_stopping_rounds=50, verbose_eval=200
)
train_time = time.time() - t0
print(f'Training time: {train_time:.1f}s')

xgb_multi_prob = xgb_multi_model.predict(dtest_m)
xgb_multi_pred = np.argmax(xgb_multi_prob, axis=1)
metrics = eval_multi(y_test_multi, xgb_multi_pred, 'XGBoost-Default-Multi')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(
    lambda x: np.argmax(xgb_multi_model.predict(x), axis=1), dtest_m, label='XGBoost-Multi'
)
RESULTS['XGB_Default_Multi'] = metrics

---
## Section 5 — Experiment 3: RandomForest & ExtraTrees Baselines

In [ ]:
# ── CELL 15: RandomForest Binary ────────────────────────────────────────────
print('=== EXP 3a: RandomForest — Binary (baseline) ===')

t0 = time.time()
rf_bin = RandomForestClassifier(
    n_estimators=200, class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_STATE
)
rf_bin.fit(X_train_np, y_train_bin)
train_time = time.time() - t0

rf_bin_val_prob  = rf_bin.predict_proba(X_val_np)[:,1]
rf_bin_threshold = tune_threshold(y_val_bin, rf_bin_val_prob)
rf_bin_prob      = rf_bin.predict_proba(X_test_np)[:,1]
metrics = eval_binary(y_test_bin, rf_bin_prob, rf_bin_threshold, 'RandomForest-Binary')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(
    lambda x: rf_bin.predict_proba(x)[:,1], X_test_np, label='RF-Binary'
)
RESULTS['RF_Binary'] = metrics

In [ ]:
# ── CELL 16: RandomForest Multi ─────────────────────────────────────────────
print('=== EXP 3b: RandomForest — Multi-Class (baseline) ===')

t0 = time.time()
rf_multi = RandomForestClassifier(
    n_estimators=200, class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_STATE
)
rf_multi.fit(X_train_np, y_train_multi)
train_time = time.time() - t0

rf_multi_pred = rf_multi.predict(X_test_np)
metrics = eval_multi(y_test_multi, rf_multi_pred, 'RandomForest-Multi')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(rf_multi.predict, X_test_np, label='RF-Multi')
RESULTS['RF_Multi'] = metrics

In [ ]:
# ── CELL 17: ExtraTrees Multi ───────────────────────────────────────────────
print('=== EXP 3c: ExtraTrees — Multi-Class ===')

t0 = time.time()
et_multi = ExtraTreesClassifier(
    n_estimators=300, class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_STATE
)
et_multi.fit(X_train_np, y_train_multi)
train_time = time.time() - t0

et_multi_pred = et_multi.predict(X_test_np)
metrics = eval_multi(y_test_multi, et_multi_pred, 'ExtraTrees-Multi')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(et_multi.predict, X_test_np, label='ET-Multi')
RESULTS['ET_Multi'] = metrics

---
## Section 6 — Experiment 4: CatBoost

In [ ]:
# ── CELL 18: CatBoost Binary + Multi ───────────────────────────────────────
if HAS_CATBOOST:
    from catboost import CatBoostClassifier, Pool

    # ── Binary
    print('=== EXP 4a: CatBoost — Binary ===')
    cb_bin = CatBoostClassifier(
        loss_function='Logloss', eval_metric='AUC',
        iterations=1500, depth=8, learning_rate=0.05,
        auto_class_weights='Balanced',
        early_stopping_rounds=50, random_seed=RANDOM_STATE, verbose=200
    )
    t0 = time.time()
    cb_bin.fit(X_train_np, y_train_bin,
               eval_set=(X_val_np, y_val_bin))
    train_time = time.time() - t0

    cb_bin_val_prob  = cb_bin.predict_proba(X_val_np)[:,1]
    cb_bin_threshold = tune_threshold(y_val_bin, cb_bin_val_prob)
    cb_bin_prob      = cb_bin.predict_proba(X_test_np)[:,1]
    metrics = eval_binary(y_test_bin, cb_bin_prob, cb_bin_threshold, 'CatBoost-Binary')
    metrics['TrainTime_s'] = round(train_time, 1)
    metrics['PredSpeed']   = benchmark_speed(
        lambda x: cb_bin.predict_proba(x)[:,1], X_test_np, label='CatBoost-Binary'
    )
    RESULTS['CatBoost_Binary'] = metrics

    # ── Multi
    print('\n=== EXP 4b: CatBoost — Multi-Class ===')
    cb_multi = CatBoostClassifier(
        loss_function='MultiClass', eval_metric='TotalF1',
        iterations=1500, depth=8, learning_rate=0.05,
        auto_class_weights='Balanced',
        early_stopping_rounds=50, random_seed=RANDOM_STATE, verbose=200
    )
    t0 = time.time()
    cb_multi.fit(X_train_np, y_train_multi,
                 eval_set=(X_val_np, y_val_multi))
    train_time = time.time() - t0

    cb_multi_pred = cb_multi.predict(X_test_np).reshape(-1).astype(int)
    metrics = eval_multi(y_test_multi, cb_multi_pred, 'CatBoost-Multi')
    metrics['TrainTime_s'] = round(train_time, 1)
    metrics['PredSpeed']   = benchmark_speed(
        lambda x: cb_multi.predict(x), X_test_np, label='CatBoost-Multi'
    )
    RESULTS['CatBoost_Multi'] = metrics
else:
    print('Skipping CatBoost (not installed).')

---
## Section 7 — Experiment 5: Optuna Hyperparameter Tuning (LightGBM)

In [ ]:
# ── CELL 19: Optuna — LightGBM Binary ──────────────────────────────────────
print(f'=== EXP 5a: Optuna LightGBM Binary ({OPTUNA_TRIALS} trials) ===')

# Subsample for speed during tuning
def _subsample(X, y, n=80_000):
    if len(X) <= n: return X, y
    idx, _ = train_test_split(np.arange(len(X)), train_size=n,
                              stratify=y, random_state=RANDOM_STATE)
    return X[idx], y.iloc[idx] if hasattr(y, 'iloc') else y[idx]

tX_tr, ty_tr = _subsample(X_train_np, y_train_bin, 80_000)
tX_vl, ty_vl = _subsample(X_val_np,   y_val_bin,   30_000)

def lgb_bin_objective(trial):
    p = {
        'objective': 'binary', 'metric': 'auc',
        'learning_rate': trial.suggest_float('lr', 0.01, 0.15, log=True),
        'num_leaves':    trial.suggest_int('num_leaves', 31, 255),
        'max_depth':     trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':     trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':    trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': pos_weight,
        'device': 'cpu', 'verbose': -1, 'seed': RANDOM_STATE
    }
    tr = lgb.Dataset(tX_tr, label=ty_tr)
    vl = lgb.Dataset(tX_vl, label=ty_vl, reference=tr)
    m  = lgb.train(p, tr, num_boost_round=500,
                   valid_sets=[vl],
                   callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(-1)])
    return roc_auc_score(ty_vl, m.predict(tX_vl))

study_bin = optuna.create_study(direction='maximize',
                                 pruner=optuna.pruners.MedianPruner())
study_bin.optimize(lgb_bin_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
best_bin_params = study_bin.best_params
print('Best params:', best_bin_params)
print('Best val AUC:', round(study_bin.best_value, 5))

In [ ]:
# ── CELL 20: Retrain LightGBM with tuned params — Binary ───────────────────
print('=== EXP 5b: LightGBM Tuned — Binary (full retrain) ===')

tuned_bin_params = {
    'objective': 'binary', 'metric': 'auc',
    'scale_pos_weight': pos_weight,
    'device': 'cpu', 'verbose': -1, 'seed': RANDOM_STATE,
    **{k: best_bin_params[k] for k in [
        'num_leaves','max_depth','min_child_samples',
        'subsample','colsample_bytree','reg_alpha','reg_lambda'
    ]},
    'learning_rate': best_bin_params['lr']
}

t0 = time.time()
lgb_tuned_bin = lgb.train(
    tuned_bin_params, lgb_bin_train, num_boost_round=2000,
    valid_sets=[lgb_bin_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0

val_prob  = lgb_tuned_bin.predict(X_val_np)
threshold = tune_threshold(y_val_bin, val_prob)
test_prob = lgb_tuned_bin.predict(X_test_np)
metrics = eval_binary(y_test_bin, test_prob, threshold, 'LightGBM-Tuned-Binary')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(lgb_tuned_bin.predict, X_test_np, label='LGB-Tuned-Binary')
RESULTS['LGB_Tuned_Binary'] = metrics

In [ ]:
# ── CELL 21: Optuna — LightGBM Multi ───────────────────────────────────────
print(f'=== EXP 5c: Optuna LightGBM Multi ({OPTUNA_TRIALS} trials) ===')

tX_tr_m, ty_tr_m = _subsample(X_train_np, y_train_multi, 80_000)
tX_vl_m, ty_vl_m = _subsample(X_val_np,   y_val_multi,   30_000)

sw_tr_m = pd.Series(ty_tr_m).map(cw).to_numpy(dtype=np.float32)
sw_vl_m = pd.Series(ty_vl_m).map(cw).to_numpy(dtype=np.float32)

def lgb_multi_objective(trial):
    p = {
        'objective': 'multiclass', 'num_class': N_CLASSES,
        'metric': 'multi_logloss',
        'learning_rate': trial.suggest_float('lr', 0.01, 0.15, log=True),
        'num_leaves':    trial.suggest_int('num_leaves', 63, 255),
        'max_depth':     trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':     trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':    trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'device': 'cpu', 'verbose': -1, 'seed': RANDOM_STATE
    }
    tr = lgb.Dataset(tX_tr_m, label=ty_tr_m, weight=sw_tr_m)
    vl = lgb.Dataset(tX_vl_m, label=ty_vl_m, weight=sw_vl_m, reference=tr)
    m  = lgb.train(p, tr, num_boost_round=500,
                   valid_sets=[vl],
                   callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(-1)])
    pred = np.argmax(m.predict(tX_vl_m), axis=1)
    return f1_score(ty_vl_m, pred, average='macro')

study_multi = optuna.create_study(direction='maximize',
                                   pruner=optuna.pruners.MedianPruner())
study_multi.optimize(lgb_multi_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
best_multi_params = study_multi.best_params
print('Best params:', best_multi_params)
print('Best val Macro-F1:', round(study_multi.best_value, 5))

In [ ]:
# ── CELL 22: Retrain LightGBM with tuned params — Multi ────────────────────
print('=== EXP 5d: LightGBM Tuned — Multi (full retrain) ===')

tuned_multi_params = {
    'objective': 'multiclass', 'num_class': N_CLASSES,
    'metric': 'multi_logloss',
    'device': 'cpu', 'verbose': -1, 'seed': RANDOM_STATE,
    **{k: best_multi_params[k] for k in [
        'num_leaves','max_depth','min_child_samples',
        'subsample','colsample_bytree','reg_alpha','reg_lambda'
    ]},
    'learning_rate': best_multi_params['lr']
}

t0 = time.time()
lgb_tuned_multi = lgb.train(
    tuned_multi_params, lgb_multi_train, num_boost_round=2000,
    valid_sets=[lgb_multi_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0

tuned_multi_prob = lgb_tuned_multi.predict(X_test_np)
tuned_multi_pred = np.argmax(tuned_multi_prob, axis=1)
metrics = eval_multi(y_test_multi, tuned_multi_pred, 'LightGBM-Tuned-Multi')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['PredSpeed']   = benchmark_speed(
    lambda x: np.argmax(lgb_tuned_multi.predict(x), axis=1), X_test_np, label='LGB-Tuned-Multi'
)
RESULTS['LGB_Tuned_Multi'] = metrics

---
## Section 8 — Experiment 6: Class Imbalance Strategies

In [ ]:
# ── CELL 23: Strategy A — No class weighting (LightGBM Multi) ──────────────
print('=== EXP 6a: LightGBM Multi — No class weights ===')

lgb_noweight_params = {**lgb_multi_params}
lgb_noweight_train = lgb.Dataset(X_train_np, label=y_train_multi, feature_name=FEATURE_NAMES)
lgb_noweight_val   = lgb.Dataset(X_val_np, label=y_val_multi, reference=lgb_noweight_train)

t0 = time.time()
lgb_noweight = lgb.train(
    lgb_noweight_params, lgb_noweight_train, num_boost_round=2000,
    valid_sets=[lgb_noweight_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0

pred = np.argmax(lgb_noweight.predict(X_test_np), axis=1)
metrics = eval_multi(y_test_multi, pred, 'LGB-Multi-NoWeights')
metrics['TrainTime_s'] = round(train_time, 1)
RESULTS['LGB_Multi_NoWeights'] = metrics

In [ ]:
# ── CELL 24: Strategy B — SMOTE on subsample (LightGBM Multi) ──────────────
print('=== EXP 6b: LightGBM Multi — SMOTE oversampling ===')

# Subsample before SMOTE to avoid memory explosion
sm_idx, _ = train_test_split(
    np.arange(len(X_train_np)), train_size=min(SMOTE_SUBSAMPLE, len(X_train_np)),
    stratify=y_train_multi, random_state=RANDOM_STATE
)
X_sm_in = X_train_np[sm_idx]
y_sm_in = y_train_multi.iloc[sm_idx].to_numpy()

# SMOTE requires at least k+1 samples per class
class_counts = pd.Series(y_sm_in).value_counts()
k_neighbors  = max(1, min(5, class_counts.min() - 1))
print(f'  Rarest class has {class_counts.min()} samples → k_neighbors={k_neighbors}')

try:
    smote = SMOTE(k_neighbors=k_neighbors, random_state=RANDOM_STATE, n_jobs=-1)
    X_smote, y_smote = smote.fit_resample(X_sm_in, y_sm_in)
    print(f'  SMOTE: {len(X_sm_in):,} → {len(X_smote):,} samples')

    lgb_smote_train = lgb.Dataset(X_smote, label=y_smote, feature_name=FEATURE_NAMES)
    lgb_smote_val   = lgb.Dataset(X_val_np, label=y_val_multi, reference=lgb_smote_train)

    t0 = time.time()
    lgb_smote = lgb.train(
        lgb_multi_params, lgb_smote_train, num_boost_round=1500,
        valid_sets=[lgb_smote_val],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
    )
    train_time = time.time() - t0

    pred = np.argmax(lgb_smote.predict(X_test_np), axis=1)
    metrics = eval_multi(y_test_multi, pred, 'LGB-Multi-SMOTE')
    metrics['TrainTime_s'] = round(train_time, 1)
    RESULTS['LGB_Multi_SMOTE'] = metrics
except Exception as e:
    print(f'SMOTE failed: {e}. Skipping.')
    RESULTS['LGB_Multi_SMOTE'] = {'Macro_F1': 'FAILED', 'Weighted_F1': 'FAILED'}

---
## Section 9 — Experiment 7: Hierarchical Pipeline

In [ ]:
# ── CELL 25: Hierarchical — Stage 1 (LGB Binary) → Stage 2 (LGB Multi) ─────
print('=== EXP 7a: Hierarchical — LGB Binary → LGB Multi ===')

# Use best LGB binary model already trained
BENIGN_IDX = int(np.where(np.array(CLASS_NAMES) == 'Benign')[0][0])

# Stage 1 prediction on test set
s1_prob = lgb_bin_model.predict(X_test_np)
s1_threshold = lgb_bin_threshold
s1_attack_mask = (s1_prob >= s1_threshold)  # flagged as attack

# Stage 2: only train on attack samples from training set
train_attack_mask = (y_train_bin.values == 1)
X_train_atk = X_train_np[train_attack_mask]
y_train_atk = y_train_multi.values[train_attack_mask]

val_attack_mask = (y_val_bin.values == 1)
X_val_atk = X_val_np[val_attack_mask]
y_val_atk = y_val_multi.values[val_attack_mask]

sw_atk_train = pd.Series(y_train_atk).map(cw).to_numpy(dtype=np.float32)
sw_atk_val   = pd.Series(y_val_atk).map(cw).to_numpy(dtype=np.float32)

# Remove benign class from attack-only stage 2 classes
atk_classes = sorted(set(y_train_atk.tolist()))
n_atk_classes = len(atk_classes)
cls_remap = {old: new for new, old in enumerate(atk_classes)}
cls_unmap = {new: old for old, new in cls_remap.items()}
y_train_atk_r = np.array([cls_remap[v] for v in y_train_atk])
y_val_atk_r   = np.array([cls_remap[v] for v in y_val_atk])

lgb_s2_params = {**lgb_multi_params, 'num_class': n_atk_classes}
lgb_s2_train = lgb.Dataset(X_train_atk, label=y_train_atk_r, weight=sw_atk_train, feature_name=FEATURE_NAMES)
lgb_s2_val   = lgb.Dataset(X_val_atk, label=y_val_atk_r, weight=sw_atk_val, reference=lgb_s2_train)

t0 = time.time()
lgb_s2 = lgb.train(
    lgb_s2_params, lgb_s2_train, num_boost_round=1500,
    valid_sets=[lgb_s2_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0

# Assemble final predictions
final_hier_pred = np.full(len(X_test_np), BENIGN_IDX, dtype=int)
if s1_attack_mask.sum() > 0:
    s2_prob_raw = lgb_s2.predict(X_test_np[s1_attack_mask])
    s2_pred_r   = np.argmax(s2_prob_raw, axis=1)
    s2_pred     = np.array([cls_unmap[v] for v in s2_pred_r])
    final_hier_pred[s1_attack_mask] = s2_pred

metrics = eval_multi(y_test_multi, final_hier_pred, 'Hierarchical-LGB→LGB')
metrics['TrainTime_s'] = round(train_time, 1)
RESULTS['Hierarchical_LGB_LGB'] = metrics
print(f'  Routed to Stage 2: {s1_attack_mask.sum():,}/{len(X_test_np):,} samples')

In [ ]:
# ── CELL 26: Hierarchical — ExtraTrees → LGB Multi ─────────────────────────
print('=== EXP 7b: Hierarchical — ExtraTrees Stage1 → LGB Multi Stage2 ===')

t0 = time.time()
et_s1 = ExtraTreesClassifier(
    n_estimators=300, class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_STATE
)
et_s1.fit(X_train_np, y_train_bin)

et_s1_val_prob = et_s1.predict_proba(X_val_np)[:,1]
et_s1_thr = tune_threshold(y_val_bin, et_s1_val_prob)
et_s1_test_prob = et_s1.predict_proba(X_test_np)[:,1]
et_attack_mask  = (et_s1_test_prob >= et_s1_thr)

final_et_hier = np.full(len(X_test_np), BENIGN_IDX, dtype=int)
if et_attack_mask.sum() > 0:
    s2_prob_raw = lgb_s2.predict(X_test_np[et_attack_mask])
    s2_pred_r   = np.argmax(s2_prob_raw, axis=1)
    s2_pred     = np.array([cls_unmap[v] for v in s2_pred_r])
    final_et_hier[et_attack_mask] = s2_pred

train_time = time.time() - t0
metrics = eval_multi(y_test_multi, final_et_hier, 'Hierarchical-ET→LGB')
metrics['TrainTime_s'] = round(train_time, 1)
RESULTS['Hierarchical_ET_LGB'] = metrics
print(f'  Routed to Stage 2: {et_attack_mask.sum():,}/{len(X_test_np):,} samples')

---
## Section 10 — Experiment 8: Feature Ablation (Causal vs Full)

In [ ]:
# ── CELL 27: Identify post-flow / leakage-prone features ────────────────────
LEAKAGE_KEYWORDS = [
    'flow_duration', 'flow_iat', 'flow_bytes', 'flow_packets',
    'packet_length_variance', 'packet_length_std',
    'idle_', 'active_', 'flow_bytes/s', 'flow_packets/s'
]

leaky_cols  = [c for c in FEATURE_NAMES if any(k in c for k in LEAKAGE_KEYWORDS)]
causal_cols = [c for c in FEATURE_NAMES if c not in leaky_cols]

print(f'Total features:   {len(FEATURE_NAMES)}')
print(f'Leakage-prone:    {len(leaky_cols)}')
print(f'Causal (safe):    {len(causal_cols)}')
print('\nLeakage features:', leaky_cols)

In [ ]:
# ── CELL 28: Top-20 feature subset (by LGB gain) ────────────────────────────
lgb_gain = lgb_multi_model.feature_importance(importance_type='gain')
feat_imp_df = pd.DataFrame({'feature': FEATURE_NAMES, 'gain': lgb_gain})\
              .sort_values('gain', ascending=False)
top20_features = feat_imp_df['feature'].head(20).tolist()
print('Top 20 features by gain:', top20_features)

In [ ]:
# ── CELL 29: Ablation — Causal-only features, LightGBM Multi ────────────────
print('=== EXP 8a: LightGBM Multi — Causal features only ===')

causal_idx = [FEATURE_NAMES.index(c) for c in causal_cols]
X_tr_c = X_train_np[:, causal_idx]
X_vl_c = X_val_np[:, causal_idx]
X_te_c = X_test_np[:, causal_idx]

lgb_causal_train = lgb.Dataset(X_tr_c, label=y_train_multi, weight=sw_train, feature_name=causal_cols)
lgb_causal_val   = lgb.Dataset(X_vl_c, label=y_val_multi, weight=sw_val, reference=lgb_causal_train)

t0 = time.time()
lgb_causal = lgb.train(
    lgb_multi_params, lgb_causal_train, num_boost_round=2000,
    valid_sets=[lgb_causal_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0

pred = np.argmax(lgb_causal.predict(X_te_c), axis=1)
metrics = eval_multi(y_test_multi, pred, 'LGB-Multi-CausalFeatures')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['n_features'] = len(causal_cols)
RESULTS['LGB_Multi_Causal'] = metrics

In [ ]:
# ── CELL 30: Ablation — Top-20 features only ────────────────────────────────
print('=== EXP 8b: LightGBM Multi — Top 20 features only ===')

top20_idx = [FEATURE_NAMES.index(c) for c in top20_features]
X_tr_t = X_train_np[:, top20_idx]
X_vl_t = X_val_np[:, top20_idx]
X_te_t = X_test_np[:, top20_idx]

lgb_top20_train = lgb.Dataset(X_tr_t, label=y_train_multi, weight=sw_train, feature_name=top20_features)
lgb_top20_val   = lgb.Dataset(X_vl_t, label=y_val_multi, weight=sw_val, reference=lgb_top20_train)

t0 = time.time()
lgb_top20 = lgb.train(
    lgb_multi_params, lgb_top20_train, num_boost_round=2000,
    valid_sets=[lgb_top20_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)]
)
train_time = time.time() - t0

pred = np.argmax(lgb_top20.predict(X_te_t), axis=1)
metrics = eval_multi(y_test_multi, pred, 'LGB-Multi-Top20Features')
metrics['TrainTime_s'] = round(train_time, 1)
metrics['n_features'] = 20
RESULTS['LGB_Multi_Top20'] = metrics

---
## Section 11 — Experiment 9: Probability Calibration for Rare Classes

In [ ]:
# ── CELL 31: Adaptive per-class threshold tuning ────────────────────────────
print('=== EXP 9a: Adaptive Per-Class Thresholds (LGB Multi) ===')

val_prob_multi = lgb_multi_model.predict(X_val_np)
test_prob_multi = lgb_multi_model.predict(X_test_np)

rare_class_ids = [
    i for i in range(N_CLASSES)
    if (y_train_multi == i).mean() < 0.01 and i != BENIGN_IDX
]
print('Rare classes (<1%):', [CLASS_NAMES[i] for i in rare_class_ids])

adaptive_thresholds = {}
for cls_idx in rare_class_ids:
    y_bin_v = (y_val_multi.values == cls_idx).astype(int)
    if y_bin_v.sum() == 0:
        adaptive_thresholds[cls_idx] = 1.0
        continue
    p, r, t = precision_recall_curve(y_bin_v, val_prob_multi[:, cls_idx])
    if len(t) == 0:
        adaptive_thresholds[cls_idx] = 1.0
        continue
    cdf = pd.DataFrame({'threshold': t, 'precision': p[:-1], 'recall': r[:-1]})
    feasible = cdf[cdf['precision'] >= 0.30]
    if not feasible.empty:
        row = feasible.sort_values(['recall','precision'], ascending=[False,False]).iloc[0]
    else:
        cdf['f1'] = 2*cdf['precision']*cdf['recall'] / (cdf['precision']+cdf['recall']+1e-12)
        row = cdf.sort_values('f1', ascending=False).iloc[0]
    adaptive_thresholds[cls_idx] = float(row['threshold'])

print('Adaptive thresholds:', {CLASS_NAMES[k]: round(v, 4) for k, v in adaptive_thresholds.items()})

# Apply adaptive thresholds
base_pred = np.argmax(test_prob_multi, axis=1).copy()
for i in range(len(base_pred)):
    for cls_idx, thr in adaptive_thresholds.items():
        if test_prob_multi[i, cls_idx] >= thr:
            base_pred[i] = cls_idx
            break

metrics = eval_multi(y_test_multi, base_pred, 'LGB-Multi-AdaptiveThresh')
RESULTS['LGB_Multi_AdaptiveThresh'] = metrics

In [ ]:
# ── CELL 32: Probability calibration (sigmoid) — Binary ─────────────────────
print('=== EXP 9b: Probability Calibration (sigmoid) — LGB Binary ===')

try:
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.base import BaseEstimator, ClassifierMixin

    # Wrap LGB model for sklearn interface
    class LGBWrapper(BaseEstimator, ClassifierMixin):
        def __init__(self, model): self.model = model
        def fit(self, X, y): return self
        def predict(self, X): return (self.model.predict(X) > 0.5).astype(int)
        def predict_proba(self, X):
            p = self.model.predict(X)
            return np.column_stack([1-p, p])
        def classes_(self): return np.array([0, 1])

    lgb_wrap = LGBWrapper(lgb_bin_model)
    lgb_wrap.classes_ = np.array([0, 1])

    cal_model = CalibratedClassifierCV(estimator=lgb_wrap, method='sigmoid', cv='prefit')
    cal_model.fit(X_val_np, y_val_bin)

    cal_prob = cal_model.predict_proba(X_test_np)[:,1]
    cal_thr  = tune_threshold(y_val_bin, cal_model.predict_proba(X_val_np)[:,1])
    metrics  = eval_binary(y_test_bin, cal_prob, cal_thr, 'LGB-Binary-Calibrated')
    RESULTS['LGB_Binary_Calibrated'] = metrics
except Exception as e:
    print(f'Calibration failed: {e}')
    RESULTS['LGB_Binary_Calibrated'] = {'AUC': 'FAILED'}

---
## Section 12 — Experiment 10: Threshold Sweep (Binary)

In [ ]:
# ── CELL 33: Threshold sweep — Precision/Recall tradeoff ────────────────────
print('=== EXP 10: Threshold Sweep (LGB Binary) ===')

thresholds = np.arange(0.1, 0.91, 0.05)
sweep_results = []
test_prob = lgb_bin_model.predict(X_test_np)

for thr in thresholds:
    pred = (test_prob >= thr).astype(int)
    sweep_results.append({
        'Threshold': round(thr, 2),
        'Precision': round(precision_score(y_test_bin, pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test_bin, pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test_bin, pred, zero_division=0), 4),
    })

sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.plot(sweep_df['Threshold'], sweep_df['Precision'], 'b-o', label='Precision', markersize=4)
plt.plot(sweep_df['Threshold'], sweep_df['Recall'],    'r-s', label='Recall', markersize=4)
plt.plot(sweep_df['Threshold'], sweep_df['F1'],        'g-^', label='F1', markersize=4)
plt.axvline(lgb_bin_threshold, color='gray', linestyle='--', label=f'Selected={lgb_bin_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('LightGBM Binary — Threshold vs Precision / Recall / F1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_sweep.png', dpi=120)
plt.show()

---
## Section 13 — Feature Importance Plots

In [ ]:
# ── CELL 34: Feature importance — LGB vs XGB (Binary + Multi) ──────────────
print('=== Feature Importance Comparison ===')

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

def plot_importance(ax, model, title, top_n=20, kind='lgb'):
    if kind == 'lgb':
        gain  = model.feature_importance(importance_type='gain')
        names = FEATURE_NAMES
    else:
        score = model.get_score(importance_type='gain')
        gain  = [score.get(f, 0) for f in FEATURE_NAMES]
        names = FEATURE_NAMES
    df_imp = pd.DataFrame({'feature': names, 'gain': gain}).nlargest(top_n, 'gain')
    ax.barh(range(len(df_imp)), df_imp['gain'], color='steelblue' if kind=='lgb' else 'salmon')
    ax.set_yticks(range(len(df_imp)))
    ax.set_yticklabels(df_imp['feature'], fontsize=8)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel('Gain')

plot_importance(axes[0,0], lgb_bin_model,   'LightGBM — Binary (Top 20)', kind='lgb')
plot_importance(axes[0,1], xgb_bin_model,   'XGBoost — Binary (Top 20)', kind='xgb')
plot_importance(axes[1,0], lgb_multi_model, 'LightGBM — Multi (Top 20)', kind='lgb')
plot_importance(axes[1,1], xgb_multi_model, 'XGBoost — Multi (Top 20)', kind='xgb')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

---
## Section 14 — Confusion Matrices

In [ ]:
# ── CELL 35: Confusion matrices — Binary ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (label, prob, thr) in zip(axes, [
    ('LightGBM', lgb_bin_model.predict(X_test_np), lgb_bin_threshold),
    ('XGBoost',  xgb_bin_model.predict(dtest),      xgb_bin_threshold),
]):
    pred = (prob >= thr).astype(int)
    cm   = confusion_matrix(y_test_bin, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{label} — Binary CM')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('cm_binary.png', dpi=120)
plt.show()

In [ ]:
# ── CELL 36: Confusion matrices — Multi-Class (normalized) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (label, pred, cmap) in zip(axes, [
    ('LightGBM', lgb_multi_pred,  'Blues'),
    ('XGBoost',  xgb_multi_pred,  'Greens'),
]):
    cm = confusion_matrix(y_test_multi, pred, normalize='true')
    sns.heatmap(cm, ax=ax, cmap=cmap, vmin=0, vmax=1,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                annot=True, fmt='.2f', annot_kws={'size': 7})
    ax.set_title(f'{label} — Multi-Class Recall Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('cm_multiclass.png', dpi=120)
plt.show()

---
## Section 15 — MASTER RESULTS TABLE & CONCLUSION

In [ ]:
# ── CELL 37: Master Results Table ───────────────────────────────────────────
print('=' * 80)
print('MASTER RESULTS — ALL EXPERIMENTS')
print('=' * 80)

rows = []
for exp_name, metrics in RESULTS.items():
    row = {'Experiment': exp_name}
    row.update(metrics)
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('Experiment')

# Split binary vs multi
binary_cols = ['AUC', 'F1', 'Recall', 'Precision', 'Threshold', 'TrainTime_s', 'PredSpeed']
multi_cols  = ['Macro_F1', 'Weighted_F1', 'TrainTime_s', 'PredSpeed']

binary_rows = [k for k in RESULTS if any(c in RESULTS[k] for c in ['AUC','F1'])]
multi_rows  = [k for k in RESULTS if 'Macro_F1' in RESULTS[k]]

print('\n── BINARY DETECTION RESULTS ──────────────────────────────────────────────')
bin_df = results_df.loc[[r for r in binary_rows if r in results_df.index],
                          [c for c in binary_cols if c in results_df.columns]]
print(bin_df.to_string())

print('\n── MULTI-CLASS RESULTS ───────────────────────────────────────────────────')
multi_df = results_df.loc[[r for r in multi_rows if r in results_df.index],
                            [c for c in multi_cols if c in results_df.columns]]
print(multi_df.to_string())

# Save to CSV
results_df.to_csv('all_experiment_results.csv')
print('\nResults saved to all_experiment_results.csv')

In [ ]:
# ── CELL 38: Visual summary — Binary models ─────────────────────────────────
bin_plot_df = bin_df[['AUC','F1','Recall','Precision']].copy()
bin_plot_df = bin_plot_df.apply(pd.to_numeric, errors='coerce').dropna()

ax = bin_plot_df.plot(kind='bar', figsize=(14, 6), rot=45, width=0.7)
ax.set_title('Binary Detection — All Experiments Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0.8, 1.01)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('summary_binary.png', dpi=120)
plt.show()

In [ ]:
# ── CELL 39: Visual summary — Multi-class models ─────────────────────────────
multi_plot_df = multi_df[['Macro_F1','Weighted_F1']].copy()
multi_plot_df = multi_plot_df.apply(pd.to_numeric, errors='coerce').dropna()

ax = multi_plot_df.plot(kind='bar', figsize=(14, 6), rot=45, width=0.7)
ax.set_title('Multi-Class Attack Classification — All Experiments Comparison')
ax.set_ylabel('F1 Score')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('summary_multiclass.png', dpi=120)
plt.show()

In [ ]:
# ── CELL 40: Speed benchmark summary ────────────────────────────────────────
print('=== INFERENCE SPEED BENCHMARK (pred/sec) ===')
speed_data = {
    k: v.get('PredSpeed', None) for k, v in RESULTS.items() if v.get('PredSpeed')
}
speed_df = pd.DataFrame.from_dict(speed_data, orient='index', columns=['pred_per_sec'])\
             .sort_values('pred_per_sec', ascending=False)
print(speed_df.to_string())
print(f'\nTarget: 1,000 pred/sec')
above = speed_df[speed_df['pred_per_sec'] >= 1000]
print(f'Models meeting real-time threshold: {len(above)}/{len(speed_df)}')
print(above.to_string())

ax = speed_df['pred_per_sec'].plot(kind='barh', figsize=(10, 6))
ax.axvline(1000, color='red', linestyle='--', label='Real-time threshold (1000/s)')
ax.set_title('Inference Speed (predictions/sec)')
ax.set_xlabel('pred/sec')
ax.legend()
plt.tight_layout()
plt.savefig('speed_benchmark.png', dpi=120)
plt.show()

In [ ]:
# ── CELL 41: Feature ablation impact ────────────────────────────────────────
print('=== FEATURE ABLATION IMPACT ===')
ablation_keys = ['LGB_Default_Multi', 'LGB_Multi_Causal', 'LGB_Multi_Top20']
abl_data = []
for k in ablation_keys:
    if k in RESULTS:
        r = RESULTS[k]
        abl_data.append({
            'Experiment': k,
            'n_features': r.get('n_features', len(FEATURE_NAMES)),
            'Macro_F1':   r.get('Macro_F1', None),
            'Weighted_F1': r.get('Weighted_F1', None)
        })
abl_df = pd.DataFrame(abl_data).set_index('Experiment')
print(abl_df.to_string())

full_macro = RESULTS.get('LGB_Default_Multi', {}).get('Macro_F1', 0) or 0
for k in ['LGB_Multi_Causal','LGB_Multi_Top20']:
    if k in RESULTS and 'Macro_F1' in RESULTS[k]:
        try:
            drop = (float(full_macro) - float(RESULTS[k]['Macro_F1'])) / float(full_macro) * 100
            print(f'{k}: Macro F1 drop = {drop:.2f}%')
        except Exception:
            pass

In [ ]:
# ── CELL 42: Hierarchical vs Flat comparison ─────────────────────────────────
print('=== FLAT vs HIERARCHICAL PIPELINE ===')
hier_keys = ['LGB_Default_Multi', 'LGB_Tuned_Multi', 'Hierarchical_LGB_LGB', 'Hierarchical_ET_LGB']
hier_data = []
for k in hier_keys:
    if k in RESULTS:
        r = RESULTS[k]
        hier_data.append({'Experiment': k,
                          'Macro_F1':    r.get('Macro_F1'),
                          'Weighted_F1': r.get('Weighted_F1')})
print(pd.DataFrame(hier_data).set_index('Experiment').to_string())

In [ ]:
# ── CELL 43: Automated Conclusion ────────────────────────────────────────────
print('=' * 80)
print('AUTOMATED CONCLUSION')
print('=' * 80)

def safe_float(v):
    try: return float(v)
    except: return -1.0

# ── Binary winner
bin_candidates = {k: v for k, v in RESULTS.items() if 'AUC' in v}
best_bin_key = max(bin_candidates, key=lambda k: safe_float(bin_candidates[k].get('F1', -1)))
best_bin = bin_candidates[best_bin_key]

# ── Multi winner
multi_candidates = {k: v for k, v in RESULTS.items() if 'Macro_F1' in v}
best_multi_key = max(multi_candidates, key=lambda k: safe_float(multi_candidates[k].get('Macro_F1', -1)))
best_multi = multi_candidates[best_multi_key]

# ── Speed check
rt_models = [k for k, v in RESULTS.items() if safe_float(v.get('PredSpeed', 0)) >= 1000]

# ── Feature ablation delta
full_macro_f1   = safe_float(RESULTS.get('LGB_Default_Multi', {}).get('Macro_F1', 0))
causal_macro_f1 = safe_float(RESULTS.get('LGB_Multi_Causal', {}).get('Macro_F1', 0))
top20_macro_f1  = safe_float(RESULTS.get('LGB_Multi_Top20',  {}).get('Macro_F1', 0))

print(f"""
┌─────────────────────────────────────────────────────────────────────────┐
│  TASK 1 — BINARY ANOMALY DETECTION                                      │
├─────────────────────────────────────────────────────────────────────────┤
│  Best model : {best_bin_key:<57}│
│  AUC        : {str(best_bin.get('AUC','?')):<57}│
│  F1         : {str(best_bin.get('F1','?')):<57}│
│  Recall     : {str(best_bin.get('Recall','?')):<57}│
│  Precision  : {str(best_bin.get('Precision','?')):<57}│
│  Threshold  : {str(best_bin.get('Threshold','?')):<57}│
├─────────────────────────────────────────────────────────────────────────┤
│  TASK 2 — MULTI-CLASS ATTACK CLASSIFICATION                             │
├─────────────────────────────────────────────────────────────────────────┤
│  Best model  : {best_multi_key:<56}│
│  Macro F1    : {str(best_multi.get('Macro_F1','?')):<56}│
│  Weighted F1 : {str(best_multi.get('Weighted_F1','?')):<56}│
├─────────────────────────────────────────────────────────────────────────┤
│  REAL-TIME ELIGIBILITY (>=1000 pred/sec)                                │
├─────────────────────────────────────────────────────────────────────────┤""")
for m in rt_models:
    spd = RESULTS[m].get('PredSpeed', '?')
    print(f'│  ✓ {m:<69} {str(spd):>6}/s │'[:77] + '│')
if not rt_models:
    print('│  ✗ No model meets the 1000 pred/sec threshold on this hardware         │')

print(f"""
├─────────────────────────────────────────────────────────────────────────┤
│  FEATURE ABLATION IMPACT                                                │
├─────────────────────────────────────────────────────────────────────────┤
│  Full features  Macro F1 : {str(round(full_macro_f1,4)):<49}│
│  Causal-only    Macro F1 : {str(round(causal_macro_f1,4)):<49}│
│  Top-20 only    Macro F1 : {str(round(top20_macro_f1,4)):<49}│
│  Causal drop   : {str(round((full_macro_f1 - causal_macro_f1)/max(full_macro_f1,1e-6)*100, 2))+'%':<58}│
│  Top-20 drop   : {str(round((full_macro_f1 - top20_macro_f1)/max(full_macro_f1,1e-6)*100, 2))+'%':<58}│
├─────────────────────────────────────────────────────────────────────────┤
│  RECOMMENDATION FOR YOUR DEEP LEARNING MODEL                            │
├─────────────────────────────────────────────────────────────────────────┤
│  1. Use LightGBM binary scores as hard baseline to beat (AUC / Recall)  │
│  2. Macro F1 on multi-class is your key metric — rare classes matter     │
│  3. If causal drop > 5%, avoid leakage features in DL too               │
│  4. Target recall >= 0.95 on binary detection as minimum bar            │
│  5. Hierarchical pipeline (binary first → family second) worth testing  │
│     in DL too (binary head + multi-class head)                          │
└─────────────────────────────────────────────────────────────────────────┘""")

print('\nAll done! Check all_experiment_results.csv and the PNG plots.')